# 00 — Conferir o ambiente

Rode este notebook primeiro. Ele confirma que MinIO, Nessie e Dremio respondem, mostra as zonas
configuradas e lista o que ja existe no catalogo.

Se algo aqui falhar, nao adianta seguir para os notebooks de origem.

In [ ]:
import os, urllib.request
from lakehouse import ZONAS, MINIO_ENDPOINT, NESSIE_URI

print("zonas configuradas:")
for papel, bucket in ZONAS.items():
    print(f"  {papel:<10} -> {bucket}")
print(f"\nMinIO  : {MINIO_ENDPOINT}")
print(f"Nessie : {NESSIE_URI}")

## Os servicos respondem?

Endpoints usam **nome de servico** porque este notebook roda dentro da rede Docker.
Do seu navegador, os enderecos sao `localhost:<porta publicada>`.

In [ ]:
def checar(nome, url):
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            print(f"  {nome:<8} OK  ({r.status})")
    except Exception as e:
        print(f"  {nome:<8} FALHOU: {e}")

checar("MinIO",  f"{MINIO_ENDPOINT}/minio/health/live")
checar("Nessie", f"{NESSIE_URI}/config")
checar("Dremio", "http://dremio:9047")

## Subir a sessao Spark

A primeira execucao baixa os JARs do Maven Central: leva de 1 a 3 minutos e **exige internet**.
Depois disso ficam em cache no `~/.ivy2` do container e as proximas sao rapidas.

Existe **uma sessao por kernel**. Se precisar mudar a lista de pacotes, reinicie o kernel.

In [ ]:
from lakehouse import sessao, listar

spark = sessao("00-ambiente")
print("Spark", spark.version, "| master:", spark.sparkContext.master)

In [ ]:
print("tabelas ja existentes no catalogo:\n")
listar(spark)

## Arquivos na zona de entrada

E de onde os notebooks de CSV e JSON leem. Para subir arquivos novos, use o console do MinIO
(<http://localhost:9001>) ou coloque em `seed/minio-data/` antes do primeiro boot.

In [ ]:
import boto3
s3 = boto3.client("s3", endpoint_url=MINIO_ENDPOINT,
                  aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
                  aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"])
resp = s3.list_objects_v2(Bucket=ZONAS["entrada"])
for obj in resp.get("Contents", []):
    print(f"  {obj['Key']:<40} {obj['Size']:>8} bytes")
else:
    if not resp.get("Contents"):
        print("  (zona de entrada vazia)")